# Final NIFTY50 Flash Crash Dataset Builder

This notebook prepares a machine learning dataset for flash crash prediction.

Steps:
1. Load all NIFTY50 stock CSV files
2. Merge them into one dataset
3. Clean and sort data
4. Create financial features
5. Generate realistic crash labels
6. Normalize features
7. Save the final dataset


In [59]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler

## Step 1: Set Dataset Folder

In [60]:
# Change this path to your dataset folder
folder_path = "stk"

## Step 2: Load and Merge All Stock Files

In [61]:
all_data = []

for file in os.listdir(folder_path):
    if file.endswith('.csv') and file not in ['stock_metadata.csv', 'NIFTY50_all.csv']:
        path = os.path.join(folder_path, file)
        df = pd.read_csv(path)
        ticker = file.replace('.csv','')
        df['ticker'] = ticker
        all_data.append(df)

data = pd.concat(all_data, ignore_index=True)

print('Dataset shape:', data.shape)
data.head()

Dataset shape: (236530, 23)


C:\Users\kavan\AppData\Local\Temp\ipykernel_14984\350044017.py:11: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data = pd.concat(all_data, ignore_index=True)


,Date,Symbol,Series,Prev Close,Open,High,Low,Last,Close,VWAP,...,Deliverable Volume,%Deliverble,ticker,age,sex,bmi,children,smoker,region,charges
0,2007-11-27,MUNDRAPORT,EQ,440.00,770.00,1050.00,770.0,959.0,962.90,984.72,...,9859619.0,0.3612,ADANIPORTS,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2007-11-28,MUNDRAPORT,EQ,962.90,984.00,990.00,874.0,885.0,893.90,941.38,...,1453278.0,0.3172,ADANIPORTS,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2007-11-29,MUNDRAPORT,EQ,893.90,909.00,914.75,841.0,887.0,884.20,888.09,...,1069678.0,0.2088,ADANIPORTS,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2007-11-30,MUNDRAPORT,EQ,884.20,890.00,958.00,890.0,929.0,921.55,929.17,...,1260913.0,0.2735,ADANIPORTS,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2007-12-03,MUNDRAPORT,EQ,921.55,939.75,995.00,922.0,980.0,969.30,965.65,...,816123.0,0.2741,ADANIPORTS,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 3: Clean Dataset

In [62]:
data['Date'] = pd.to_datetime(data['Date'], dayfirst=True)
data = data.sort_values(['ticker','Date'])
data = data.reset_index(drop=True)

C:\Users\kavan\AppData\Local\Temp\ipykernel_14984\1948790145.py:1: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  data['Date'] = pd.to_datetime(data['Date'], dayfirst=True)


## Step 4: Feature Engineering

In [63]:
data['return'] = data.groupby('ticker')['Close'].pct_change()

data['volatility'] = (
    data.groupby('ticker')['return']
    .rolling(10)
    .std()
    .reset_index(level=0, drop=True)
)

data['momentum'] = data['Close'] - data.groupby('ticker')['Close'].shift(5)

data['volume_change'] = data.groupby('ticker')['Volume'].pct_change()

data['vwap_diff'] = (data['Close'] - data['VWAP']) / data['VWAP']

data['high_low_spread'] = (data['High'] - data['Low']) / data['Close']

data['open_close_return'] = (data['Close'] - data['Open']) / data['Open']

data['turnover_change'] = data.groupby('ticker')['Turnover'].pct_change()

C:\Users\kavan\AppData\Local\Temp\ipykernel_14984\2391939635.py:1: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  data['return'] = data.groupby('ticker')['Close'].pct_change()
C:\Users\kavan\AppData\Local\Temp\ipykernel_14984\2391939635.py:12: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  data['volume_change'] = data.groupby('ticker')['Volume'].pct_change()
C:\Users\kavan\AppData\Local\Temp\ipykernel_14984\2391939635.py:20: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-lead

## Step 5: Crash Label Generation

In [64]:
data['price_drop'] = (
    data['Close'] - data.groupby('ticker')['Close'].shift(3)
) / data.groupby('ticker')['Close'].shift(3)

data['crash_label'] = (
    (data['price_drop'] < -0.10) &
    (data['volatility'] > data['volatility'].quantile(0.9))
).astype(int)

## Step 6: Remove Unnecessary Columns

In [65]:
data = data.drop(columns=['Symbol','Series','Prev Close','Last'], errors='ignore')

## Step 7: Handle Missing Values

In [66]:
important_cols = [
"Open","High","Low","Close","Volume","VWAP",
"return","volatility","momentum",
"volume_change","vwap_diff",
"high_low_spread","open_close_return","turnover_change"
]

data = data.dropna(subset=important_cols).reset_index(drop=True)
print('Dataset shape after cleaning:', data.shape)

Dataset shape after cleaning: (234702, 29)


In [67]:
data

,Date,Open,High,Low,Close,VWAP,Volume,Turnover,Trades,Deliverable Volume,...,return,volatility,momentum,volume_change,vwap_diff,high_low_spread,open_close_return,turnover_change,price_drop,crash_label
0,2007-12-11,1081.0,1089.00,1041.00,1047.65,1067.80,810464.0,8.654156e+13,NaN,415191.0,...,-0.025804,0.044204,6.20,-0.199423,-0.018871,0.045817,-0.030851,-0.208746,-0.031120,0
1,2007-12-12,1032.0,1065.00,1016.00,1036.80,1043.92,744799.0,7.775137e+13,NaN,363848.0,...,-0.010357,0.035017,-45.65,-0.081021,-0.006820,0.047261,0.004651,-0.101572,-0.059507,0
2,2007-12-13,1040.0,1150.00,1030.25,1129.95,1109.09,3067687.0,3.402339e+14,NaN,1040076.0,...,0.089844,0.040631,48.65,3.118812,0.018808,0.105978,0.086490,3.375922,0.050725,0
3,2007-12-14,1139.9,1140.00,1101.10,1110.50,1119.55,1070737.0,1.198746e+14,NaN,525239.0,...,-0.017213,0.042236,8.10,-0.650963,-0.008084,0.035029,-0.025792,-0.647670,0.059991,0
4,2007-12-17,1140.0,1168.00,1021.50,1044.25,1102.42,1404955.0,1.548848e+14,NaN,670298.0,...,-0.059658,0.047208,-31.15,0.312138,-0.052766,0.140292,-0.083991,0.292056,0.007186,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234697,2021-04-26,190.6,191.10,185.10,186.40,187.35,8542755.0,1.600451e+14,52374.0,2340188.0,...,-0.008511,0.046773,-7.55,0.001561,-0.005071,0.032189,-0.022036,-0.012617,-0.055724,0
234698,2021-04-27,188.0,192.95,186.80,188.15,189.41,14247767.0,2.698636e+14,73673.0,5425957.0,...,0.009388,0.046190,-2.20,0.667819,-0.006652,0.032687,0.000798,0.686172,-0.021581,0
234699,2021-04-28,188.8,190.60,187.10,189.10,188.85,8429439.0,1.591917e+14,44056.0,2413974.0,...,0.005049,0.023571,-8.30,-0.408368,0.001324,0.018509,0.001589,-0.410103,0.005851,0
234700,2021-04-29,190.8,191.65,186.00,186.55,187.44,9483009.0,1.777471e+14,60932.0,2744472.0,...,-0.013485,0.020859,-5.75,0.124987,-0.004748,0.030287,-0.022275,0.116560,0.000805,0


## Step 8: Normalize Features

In [68]:
print("Rows before scaling:", len(data))

Rows before scaling: 234702


In [69]:
features = [
'Open','High','Low','Close','Volume','VWAP',
'return','volatility','momentum','volume_change','vwap_diff',
'high_low_spread','open_close_return','turnover_change'
]

scaler = StandardScaler()
data[features] = scaler.fit_transform(data[features])

## Step 9: Final Dataset Structure

In [70]:
data = data[[
'Date','ticker','Open','High','Low','Close','Volume','VWAP',
'return','volatility','momentum','volume_change','vwap_diff',
'high_low_spread','open_close_return','turnover_change','crash_label'
]]

data.head()

,Date,ticker,Open,High,Low,Close,Volume,VWAP,return,volatility,momentum,volume_change,vwap_diff,high_low_spread,open_close_return,turnover_change,crash_label
0,2007-12-11,ADANIPORTS,-0.072516,-0.075689,-0.081371,-0.085051,-0.310769,-0.077456,-0.973551,1.376206,0.024652,-0.051911,-1.874208,0.412591,-1.202954,-0.052548,0
1,2007-12-12,ADANIPORTS,-0.091463,-0.084847,-0.091184,-0.089251,-0.319921,-0.086699,-0.405895,0.810433,-0.264672,-0.042897,-0.627410,0.469612,0.221583,-0.044444,0
2,2007-12-13,ADANIPORTS,-0.088369,-0.052412,-0.085591,-0.053190,0.003825,-0.061475,3.276124,1.156212,0.261524,0.200725,2.024327,2.788292,3.505404,0.218506,0
3,2007-12-14,ADANIPORTS,-0.049742,-0.056228,-0.057781,-0.060720,-0.274494,-0.057427,-0.657854,1.255025,0.035254,-0.086290,-0.758106,-0.013398,-0.999947,-0.085737,0
4,2007-12-17,ADANIPORTS,-0.049703,-0.045543,-0.089025,-0.086367,-0.227913,-0.064057,-2.217551,1.561236,-0.183762,-0.012963,-5.381261,4.143309,-3.335218,-0.014680,0


## Step 10: Check Label Distribution

In [71]:
print(data['crash_label'].value_counts())

crash_label
0    232228
1      2474
Name: count, dtype: int64


## Step 11: Save Final Dataset

In [72]:
data.to_csv('flash_crash_ready_dataset.csv', index=False)
print('Dataset saved as flash_crash_ready_dataset.csv')

Dataset saved as flash_crash_ready_dataset.csv
